# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# Ranking / scoring, built on top of a classifier

My lane's real question is "which pages should a reviewer look at first?" — that's a **ranking / scoring** problem, not a plain yes/no classification, even though the way I'll get the score is by training a binary classifier.

Here's the distinction I'm keeping straight: a classifier alone would just label each page "declining" or "not." What the reviewer actually needs is an ORDERED list, because they can only get through a limited number of pages per cycle. So the classifier's output (a decline probability) becomes one ingredient in a score, and the score is what gets ranked. Lane 2's own final formula in the guide does exactly this: `final_refresh_score = 100 * (0.70 * model_probability + 0.30 * normalized_baseline_score)`.

So: task type = **ranking/scoring**, using classification underneath.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Just documenting the mapping I used, from the framing-ml-problems skill's table:
task_frame = {
    "question_shape": "Which ones first?",
    "task_type": "Ranking / scoring (implemented via a classifier's probability output)",
    "typical_metric": "precision@K",
}
for k, v in task_frame.items():
    print(f"{k}: {v}")


question_shape: Which ones first?
task_type: Ranking / scoring (implemented via a classifier's probability output)
typical_metric: precision@K


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

# Target: `is_declining_label`, and it's a proxy — not an observed future outcome

For now I'm using the starter's target, `is_declining_label = (trend_direction == "down")`. But I want to be honest about what this actually is: `trend_direction` is computed by comparing `impressions_last_30d` against `impressions_prev_30d` — both numbers already sitting in the row today. So this label tells me a page **is currently** trending down, not that it **will** decline going forward. That makes it a defined-rule proxy, not an observed outcome.

This matters because of the leakage rule in the data skill: `trend_direction` and `trend_pct` are the label's own ingredients, so they can never be features — using them would let the model "predict" the label from itself.

The stronger version of this target, which I'd move to once I'm working with the full warehouse table (`fact_content_daily_performance`), is a genuine future-window label: features from the prior 90 days predicting decline over the NEXT 30 days that haven't happened yet in the training window. That's the honest target. I'm using the proxy now because it's what the starter pipeline ships and it lets me build the whole workflow end to end first.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Sketch the target exactly as the starter pipeline defines it
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(df['is_declining_label'].value_counts())
print(f"\nBase rate (share currently declining): {df['is_declining_label'].mean():.3f}")

# Confirm the leakage risk: trend_direction and trend_pct are the label's own ingredients
print("\ntrend_direction value counts (this IS the label source, never a feature):")
print(df['trend_direction'].value_counts())


is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Base rate (share currently declining): 0.542

trend_direction value counts (this IS the label source, never a feature):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

# Metric: precision@50

A reviewer isn't going to work through all 30,000 rows — they'll look at a short list. So the metric that matches the real decision is **precision@K**: of the top K pages my ranking puts first, how many are actually correct?

I'm picking **K=50** to match the starter pipeline's own reporting, so my numbers are comparable to the baseline that's already in this repo. From `outputs/model_report.md`:

| Method | Precision@50 |
|---|---:|
| baseline rules | 0.240 |
| logistic regression | 0.400 |
| decision tree | 0.540 |
| random forest | 0.740 |

"Good" for me means: does my ranking beat 0.240 (the baseline rule), by a margin big enough that it's not just noise from a 30k-row anonymized slice. ROC-AUC is a fine secondary check, but precision@50 is the number I'd actually defend to a reviewer, because it describes their real workload — not an abstract average.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Real precision@50 numbers already produced by this repo's pipeline (outputs/model_report.md)
precision_at_50 = {
    "baseline_rules": 0.240,
    "logistic_regression": 0.400,
    "decision_tree": 0.540,
    "random_forest": 0.740,
}
for method, score in precision_at_50.items():
    hits = round(score * 50)
    print(f"{method:20s} precision@50 = {score:.3f}  (~{hits} of the top 50 correct)")


baseline_rules       precision@50 = 0.240  (~12 of the top 50 correct)
logistic_regression  precision@50 = 0.400  (~20 of the top 50 correct)
decision_tree        precision@50 = 0.540  (~27 of the top 50 correct)
random_forest        precision@50 = 0.740  (~37 of the top 50 correct)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

# Unit of analysis: one pseudonymized content item, trailing 90-day snapshot

One row = one piece of published content (`content_id`) belonging to one client (`client_id`), summarized over its trailing 90 days of search and engagement data. It is NOT one client, and it is NOT one day — a client shows up many times (once per content item they own), and the daily granularity is already collapsed into 90-day totals for this starter slice.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the lane's slice using the starter pipeline's own filter, and show the real unit of analysis
lane_df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')

print(f"Rows (= content items) in my lane slice: {len(lane_df)}")
print(f"Unique clients represented: {lane_df['client_id'].nunique()}")
print(f"Median content items per client: {lane_df.groupby('client_id').size().median()}")

lane_df[['content_id', 'client_id', 'impressions_90d', 'content_age_days', 'trend_direction']].head()


Rows (= content items) in my lane slice: 30000
Unique clients represented: 32
Median content items per client: 567.0


,content_id,client_id,impressions_90d,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,141,down
3,content_331d6c4de07b,client_19581e27de,11751,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,263,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

# Why ML beats a fixed if-statement here

The starter pipeline already gives me the head-to-head evidence, so I don't have to argue this in the abstract — I can point at it:

| Method | ROC AUC | Precision@50 |
|---|---:|---:|
| baseline rules (hand-written if-statements) | 0.627 | 0.240 |
| random forest | 0.750 | 0.740 |

The rule-based baseline (`stale_visible_page`, `declining_with_demand`, `thin_visible_page`, etc.) isn't bad — it beats guessing. But it catches roughly 12 of its top 50 correctly, while the random forest catches about 37 of its top 50, on the same pages.

Why the gap? Looking at the model's own feature importances (`outputs/model_report.md`), no single signal dominates — `days_with_impressions` (0.16), `log_impressions_90d` (0.13), `avg_position` (0.11), `content_age_days` (0.10), then word count, clicks, CTR, scroll rate, all contributing smaller amounts. That's the "too messy for an if-statement" pattern: ten-plus signals, each only partially informative, combining in ways that shift depending on the other signals present. A hand-written rule can check two or three thresholds at once before it gets unreadable; a model can weigh all of them together and learn where the thresholds should actually bend for different types of pages. That's exactly where the framing skill says ML earns its place.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature importances from the repo's own trained random forest (outputs/model_report.md)
top_features = {
    "days_with_impressions": 0.1578,
    "log_impressions_90d": 0.1282,
    "avg_position": 0.1090,
    "content_age_days": 0.0955,
    "char_count": 0.0426,
    "word_count": 0.0397,
    "log_clicks_90d": 0.0346,
    "ctr": 0.0330,
    "scroll_rate": 0.0311,
    "days_with_sessions": 0.0280,
}
total_top10 = sum(top_features.values())
print(f"Top 10 features together explain {total_top10:.1%} of importance — no single feature dominates.")
for feat, imp in top_features.items():
    print(f"  {feat:25s} {imp:.4f}")


Top 10 features together explain 70.0% of importance — no single feature dominates.
  days_with_impressions     0.1578
  log_impressions_90d       0.1282
  avg_position              0.1090
  content_age_days          0.0955
  char_count                0.0426
  word_count                0.0397
  log_clicks_90d            0.0346
  ctr                       0.0330
  scroll_rate               0.0311
  days_with_sessions        0.0280


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.